# Track B — AutoGen Agentic AI 실습

**주제**: 멀티에이전트 협업 + LangGraph 협업 브릿지

**학습 목표**
- AutoGen의 대화 기반 워크플로우 이해
- GroupChat으로 역할 기반 협업 구현
- 번역 + 자기검토 루프를 에이전트 팀으로
- Track A(LangGraph) 초안을 이어받아 최종 PDF 완성 (L2 소통)

**환경**: Antigravity IDE + Colab Extension (Colab 웹 호환)

---

## 실습 순서 (Track A와의 협업)

이 노트북은 Track A(LangGraph)와 파일로 연결됩니다.

1. Track A 노트북 실행 → `workspace/reports/trackA_draft.md` 생성
2. Track B 노트북 실행 → 초안을 받아 최종 PDF 완성

**Track A 노트북 마지막에 추가할 셀** (Track A에 아직 없다면 붙여넣기):

```python
from pathlib import Path
WORKDIR = Path.cwd() / "workspace"
draft_result = report_graph.invoke(
    {"messages": [("user", "모두의연구소에 대해 조사해서 리포트 초안을 작성해줘")]},
    config=RunnableConfig(recursion_limit=15, configurable={"thread_id": "bridge-1"})
)
trackA_final_text = draft_result["messages"][-1].content
draft_path = WORKDIR / "reports" / "trackA_draft.md"
draft_path.write_text(trackA_final_text, encoding="utf-8")
print(f"Track A 초안 저장: {draft_path}")
```

## 목차
- [0] 환경 설정
- [1] AutoGen 개념 + LangGraph 비교
- [2] 2인 대화 (Coder + Executor)
- [3] GroupChat 버그수정 워크플로우
- [4] 번역 + 자기검토 GroupChat
- [5] Track A → Track B 브릿지 (핵심)
- [6] L1/L2/L3 소통 정리

## 0. 환경 설정

In [ ]:
import subprocess, sys
def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages], check=True)

pip_install(
    "pyautogen==0.2.3",
    "langchain-openai",
    "langchain-community",
    "tavily-python",
    "fpdf2",
    "pdfplumber",
    "requests",
)
print("설치 완료")

In [ ]:
import os, getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY: ")
print("키 설정 완료")

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

WORKDIR = Path.cwd() / "workspace"
WORKDIR.mkdir(exist_ok=True)
(WORKDIR / "fonts").mkdir(exist_ok=True)
(WORKDIR / "reports").mkdir(exist_ok=True)
(WORKDIR / "coding").mkdir(exist_ok=True)
print("WORKDIR:", WORKDIR.resolve())

In [ ]:
import autogen
config_list = [{
    "model": "gpt-4o-mini",
    "api_key": os.environ["OPENAI_API_KEY"],
}]
llm_config = {
    "config_list": config_list,
    "temperature": 0.0,
    "cache_seed": None,
}
print("AutoGen:", autogen.__version__)

In [ ]:
import requests
from fpdf import FPDF

FONT_PATH = WORKDIR / "fonts" / "NotoSansKR.ttf"
if not FONT_PATH.exists():
    url = "https://github.com/google/fonts/raw/main/ofl/notosanskr/NotoSansKR%5Bwght%5D.ttf"
    FONT_PATH.write_bytes(requests.get(url, timeout=60).content)
    print("폰트 다운로드 완료")

def save_pdf(content: str, filename: str = "report.pdf") -> str:
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_font("NotoSans", "", str(FONT_PATH), uni=True)
    pdf.set_font("NotoSans", size=12)
    for line in content.split("\n"):
        pdf.multi_cell(0, 8, line)
    out_path = WORKDIR / "reports" / filename
    pdf.output(str(out_path))
    return str(out_path)

print("save_pdf 준비 완료")

## 1. AutoGen 개념 + LangGraph 비교

| 관점 | LangGraph (A) | AutoGen (B) |
|------|---------------|-------------|
| 흐름 제어 | 명시적 (엣지) | 암묵적 (대화) |
| 상태 관리 | State TypedDict | chat.messages |
| emergent | 제한적 | 강함 |
| 디버깅 | 노드별 추적 | 로그 분석 |
| 러닝커브 | 그래프 개념 | 직관적 |

## 2. 2인 대화 (Coder + Executor)

In [ ]:
from autogen import AssistantAgent, UserProxyAgent

coder = AssistantAgent(
    name="Coder",
    llm_config=llm_config,
    system_message=(
        "당신은 파이썬 개발자입니다. "
        "코드는 ```python 블록으로 감싸세요. "
        "작업 완료 시 'TERMINATE'를 출력하세요."
    ),
)

executor = UserProxyAgent(
    name="Executor",
    human_input_mode="NEVER",
    code_execution_config={
        "work_dir": str(WORKDIR / "coding"),
        "use_docker": False,
    },
    max_consecutive_auto_reply=3,
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
)

executor.initiate_chat(
    coder,
    message="1부터 100까지 더하는 파이썬 코드를 작성하고 실행해서 결과를 알려줘.",
)

## 3. GroupChat 버그수정 워크플로우 (Triage / Patcher / Reviewer / Logger)

In [ ]:
from autogen import GroupChat, GroupChatManager

triage = AssistantAgent(
    name="Triage",
    llm_config=llm_config,
    system_message=(
        "당신은 버그 분류 담당자입니다. "
        "카테고리와 우선순위를 한 줄로 요약하고 'TRIAGE_DONE'으로 끝맺으세요."
    ),
)

patcher = AssistantAgent(
    name="Patcher",
    llm_config=llm_config,
    system_message=(
        "당신은 패치 개발자입니다. "
        "파이썬 패치 코드를 ```python 블록으로 작성하고 'PATCH_DONE'으로 끝맺으세요."
    ),
)

reviewer = AssistantAgent(
    name="Reviewer",
    llm_config=llm_config,
    system_message=(
        "당신은 코드 리뷰어입니다. "
        "8점 이상이면 'APPROVED', 미만이면 'REVISE'로만 답하세요."
    ),
)

logger = UserProxyAgent(
    name="Logger",
    human_input_mode="NEVER",
    code_execution_config=False,
    system_message=(
        "당신은 기록 담당자입니다. 논의를 3줄로 요약하고 'TERMINATE'를 출력하세요."
    ),
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", ""),
)

groupchat = GroupChat(
    agents=[triage, patcher, reviewer, logger],
    messages=[],
    max_round=12,
    speaker_selection_method="round_robin",
)
manager = GroupChatManager(groupchat=groupchat, llm_config=llm_config)

triage.initiate_chat(
    manager,
    message=("다음 버그를 수정하세요: '리스트에서 값을 찾을 때 "
             "len(list)-1 인덱스 접근 시 IndexError가 발생.'"),
)

## 4. 번역 + 자기검토 GroupChat (Translator / Reviewer / Retranslator)

In [ ]:
SAMPLE_EN = (
    "Hyundai Motor Company announced its Q4 2023 business results. "
    "Annual revenue increased 14.4% to KRW 162.7 trillion. "
    "Operating profit rose 54% to KRW 15.12 trillion."
)

translator = AssistantAgent(
    name="Translator",
    llm_config=llm_config,
    system_message=(
        "당신은 다국어 번역가입니다. 반드시 한국어로 번역하세요. "
        "고유명사는 '한국어(원어)' 형식. "
        "숫자/통화는 원문 + 한국식 병기. "
        "번역만 출력하고 'TRANSLATE_DONE'으로 끝맺으세요."
    ),
)

reviewer_tr = AssistantAgent(
    name="Reviewer",
    llm_config=llm_config,
    system_message=(
        "당신은 번역 검토자입니다. "
        "완전성(3)/정확성(3)/자연스러움(2)/일관성(2) 총 10점. "
        "8점 이상이면 'APPROVED', 미만이면 'REVISE'와 이유를 출력하세요."
    ),
)

retranslator = AssistantAgent(
    name="Retranslator",
    llm_config=llm_config,
    system_message=(
        "당신은 재번역가입니다. 지적된 문제를 반영해 재번역하고 'RETRANSLATE_DONE'으로 끝맺으세요."
    ),
)

groupchat_tr = GroupChat(
    agents=[translator, reviewer_tr, retranslator],
    messages=[],
    max_round=6,
    speaker_selection_method="round_robin",
)
manager_tr = GroupChatManager(groupchat=groupchat_tr, llm_config=llm_config)

translator.initiate_chat(
    manager_tr,
    message=f"다음 텍스트를 한국어로 번역하고 검토하세요:\n\n{SAMPLE_EN}",
)

## 5. Track A → Track B 브릿지 (L2 소통)

Track A(LangGraph)의 초안을 AutoGen Editor 팀이 이어받아 최종 PDF로 저장합니다.
Track A 초안 파일이 없으면 샘플 초안으로 진행합니다.

In [ ]:
draft_path = WORKDIR / "reports" / "trackA_draft.md"

if draft_path.exists():
    langgraph_draft = draft_path.read_text(encoding="utf-8")
    print(f"Track A 초안 로드: {len(langgraph_draft)}자")
else:
    langgraph_draft = (
        "# 모두의연구소 리포트 (샘플 초안)\n"
        "## 1. 개요\n"
        "모두의연구소는 2015년 설립된 커뮤니티 기반 AI 교육 스타트업입니다.\n"
        "## 2. 주요 사업\n"
        "- AI 교육 (아이펠)\n"
        "- 연구 커뮤니티 (페이퍼샵)\n"
        "## 3. 인사이트\n"
        "- 커뮤니티 기반 학습 모델의 차별성\n"
        "- 70억 원 투자 유치\n"
    )
    print("Track A 초안 없음 → 샘플 초안 사용")

print("--- 초안 앞 300자 ---")
print(langgraph_draft[:300])

In [ ]:
researcher = AssistantAgent(
    name="Researcher",
    llm_config=llm_config,
    system_message=(
        "당신은 리서치 검증자입니다. 사실 확인이 필요한 항목을 최대 3개 지적하고 'RESEARCH_DONE'으로 끝맺으세요."
    ),
)

editor = AssistantAgent(
    name="Editor",
    llm_config=llm_config,
    system_message=(
        "당신은 한국어 리포트 편집자입니다. 지적을 반영해 마크다운으로 재작성하고 'EDITOR_DONE'으로 끝맺으세요."
    ),
)

fact_checker = AssistantAgent(
    name="FactChecker",
    llm_config=llm_config,
    system_message=(
        "당신은 팩트체커입니다. 숫자/날짜/고유명사 오류를 확인하세요. "
        "8점 이상이면 'APPROVED', 미만이면 'REVISE'라고만 답하세요."
    ),
)

pdf_writer = UserProxyAgent(
    name="PDFWriter",
    human_input_mode="NEVER",
    code_execution_config={
        "work_dir": str(WORKDIR / "coding"),
        "use_docker": False,
    },
    system_message=(
        "당신은 PDF 저장 담당자입니다. APPROVED 신호 후 최종본을 PDF로 저장하고 "
        "'PDF_SAVED'와 'TERMINATE'를 출력하세요."
    ),
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", ""),
)
print("Editor 팀 구성 완료")

In [ ]:
groupchat_bridge = GroupChat(
    agents=[researcher, editor, fact_checker, pdf_writer],
    messages=[],
    max_round=10,
    speaker_selection_method="round_robin",
)
manager_bridge = GroupChatManager(groupchat=groupchat_bridge, llm_config=llm_config)

bridge_message = (
    "다음은 Track A(LangGraph)가 작성한 초안입니다. "
    "검토 → 개선 → 검증 → PDF 저장하세요.\n\n"
    f"[Track A 초안]\n{langgraph_draft}\n\n"
    "[지시]\n"
    "1. Researcher: 사실 확인 항목 지적\n"
    "2. Editor: 지적 반영해 재작성\n"
    "3. FactChecker: 8점 이상이면 APPROVED\n"
    "4. PDFWriter: bridge_final.pdf로 저장\n"
)

researcher.initiate_chat(manager_bridge, message=bridge_message)

In [ ]:
final_pdf = WORKDIR / "reports" / "bridge_final.pdf"
if final_pdf.exists():
    print(f"최종 PDF: {final_pdf} ({final_pdf.stat().st_size:,} bytes)")
else:
    print("PDF 미생성 → 수동 저장 시도")
    saved = save_pdf(langgraph_draft, "bridge_final_manual.pdf")
    print(f"대체 저장: {saved}")

In [ ]:
# 저장된 PDF가 있으면 첫 페이지 텍스트 미리보기
if final_pdf.exists():
    import pdfplumber
    with pdfplumber.open(final_pdf) as pdf:
        text = pdf.pages[0].extract_text() or ""
    print("--- PDF 1페이지 앞 500자 ---")
    print(text[:500])
else:
    print("PDF 파일이 없습니다.")

## 6. L1/L2/L3 소통 정리

### L1: 프레임워크 내부 소통 (emergent)
- Track A-4: State 공유 → 노드 순환
- Track B-2~4: GroupChat 메시지 큐 → 자동 순환

### L2: 프레임워크 간 파이프라인 소통 (오늘 실습)
- Track A(LangGraph) → `trackA_draft.md` → Track B(AutoGen)
- 파일/함수 반환값으로 배턴 터치

### L3: 실시간 양방향 소통 (개념)
- MCP / A2A / AGNTCY 프로토콜
- LangGraph: `langchain-mcp-adapters`
- AutoGen: `FastMCP`로 팀을 서버로 노출
- 별도 세션에서 다룰 것

### 강점 비교
| 강점 | LangGraph | AutoGen |
|------|:---:|:---:|
| 통제 | 강 | 약 |
| 유연성 | 약 | 강 |
| emergent | 제한 | 강 |
| 디버깅 | 쉬움 | 어려움 |

## 실습 완료

산출물: `workspace/reports/bridge_final.pdf`

다음 단계:
1. `speaker_selection_method="auto"`로 emergent 실험
2. MCP 브릿지 (L3) 별도 실습
3. Track A 자기검토 루프와 비교 분석